# 02 — Trajectory Analysis

Deep dive into ball trajectory physics and feature analysis.
This notebook explores what makes each spin type visually and quantitatively distinct.

**Key questions answered**:
1. Which features best separate spin types?
2. How early in a trajectory can we classify spin?
3. What's the effect of noise on feature stability?

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 110
print('Ready ✓')

## 1. Load Features

In [ ]:
features_path = Path('../data/processed/features.parquet')
if features_path.exists():
    df = pd.read_parquet(features_path)
    df_labeled = df[df['spin_int'] >= 0].copy()
    print(f'Loaded {len(df_labeled)} labeled samples')
    display(df_labeled.describe())
else:
    print('Features not found. Running feature engineering on synthetic data...')
    # Generate synthetic features for notebook demo
    from src.processing.feature_engineering import build_feature_vector
    import json
    
    records = []
    spins_config = {
        'topspin': lambda t: (t * 0.8 + 0.1, 0.2 + 0.35 * t**1.5),
        'backspin': lambda t: (t * 0.7 + 0.1, 0.5 - 0.1*t + 0.15*t**2),
        'sidespin': lambda t: (t * 0.8 + 0.1, 0.35 + 0.08*np.sin(t*6)),
        'float':    lambda t: (t * 0.85 + 0.1, 0.3 + 0.04*t),
    }
    LABEL_TO_INT = {'topspin': 0, 'backspin': 1, 'sidespin': 2, 'float': 3}
    
    for spin, fn in spins_config.items():
        for i in range(80):
            t = np.linspace(0, 1, 30)
            x, y = fn(t)
            x += np.random.randn(30) * 0.015
            y += np.random.randn(30) * 0.015
            positions = np.stack([x, y], axis=1).tolist()
            feats = build_feature_vector({'track_id': i, 'positions': positions, 'spin_label': spin})
            if feats:
                feats['spin_label'] = spin
                feats['spin_int'] = LABEL_TO_INT[spin]
                records.append(feats)
    
    df_labeled = pd.DataFrame(records)
    print(f'Generated {len(df_labeled)} synthetic samples')
    display(df_labeled['spin_label'].value_counts())

## 2. Feature Importance (separability)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

FEATURE_COLS = [
    'speed_mean', 'speed_std', 'speed_max', 'acc_mean', 'acc_std',
    'y_quadratic_coeff', 'lateral_drift_max', 'lateral_drift_mean',
    'curvature_mean', 'curvature_max', 'sinuosity',
    'net_y_disp', 'net_x_disp', 'hv_ratio', 'vy_sign_changes',
    'arc_length', 'chord_length', 'early_10f_y_coeff'
]
FEATURE_COLS = [c for c in FEATURE_COLS if c in df_labeled.columns]

X = df_labeled[FEATURE_COLS].fillna(0)
y = df_labeled['spin_int']

rf = RandomForestClassifier(n_estimators=100, random_state=42)
cv_scores = cross_val_score(rf, X, y, cv=5, scoring='accuracy')
print(f'Random Forest CV Accuracy: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}')

rf.fit(X, y)
importance_df = pd.DataFrame({'feature': FEATURE_COLS, 'importance': rf.feature_importances_})
importance_df = importance_df.sort_values('importance', ascending=True)

fig, ax = plt.subplots(figsize=(8, 8))
colors = ['#2196F3' if i >= len(FEATURE_COLS) - 5 else '#aaaaaa' for i in range(len(FEATURE_COLS))]
ax.barh(importance_df.feature, importance_df.importance, color=colors)
ax.set_xlabel('Feature Importance (Random Forest)')
ax.set_title('Top Features for Spin Classification')
plt.tight_layout()
plt.savefig('../reports/feature_importance.png', bbox_inches='tight')
plt.show()

## 3. PCA Visualization

In [ ]:
SPIN_COLORS = {'topspin': '#2196F3', 'backspin': '#4CAF50', 'sidespin': '#FF9800', 'float': '#9C27B0'}

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

fig, ax = plt.subplots(figsize=(9, 7))
for spin in ['topspin', 'backspin', 'sidespin', 'float']:
    mask = df_labeled['spin_label'] == spin
    ax.scatter(
        X_pca[mask, 0], X_pca[mask, 1],
        c=SPIN_COLORS[spin], label=spin.title(), alpha=0.6, s=40, edgecolors='white', linewidths=0.5
    )
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
ax.set_title('PCA of Trajectory Features')
ax.legend()
plt.tight_layout()
plt.savefig('../reports/pca_visualization.png', bbox_inches='tight')
plt.show()
print(f'Total variance explained: {sum(pca.explained_variance_ratio_):.1%}')

## 4. Early Detection — How Many Frames Do We Need?

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

# Simulate early classification by using only first N frames
frame_counts = [5, 8, 10, 15, 20, 25, 30]
accuracies = []

for n_frames in frame_counts:
    # Recompute features using only first n_frames
    from src.processing.feature_engineering import (
        compute_kinematic_features, compute_geometric_features, normalize_trajectory
    )
    
    rows = []
    for _, row in df_labeled.iterrows():
        if 'norm_traj' in row and row['norm_traj'] is not None:
            traj = np.array(row['norm_traj'])
            early = traj[:n_frames]
            if len(early) < 3:
                continue
            k = compute_kinematic_features(early)
            g = compute_geometric_features(early)
            combined = {**k, **g, 'spin_int': row['spin_int']}
            rows.append(combined)
    
    if not rows:
        accuracies.append(0)
        continue
    
    df_early = pd.DataFrame(rows).fillna(0)
    feat_cols = [c for c in df_early.columns if c != 'spin_int']
    Xe = df_early[feat_cols]
    ye = df_early['spin_int']
    
    if len(Xe) < 10:
        accuracies.append(0)
        continue
    
    gb = GradientBoostingClassifier(n_estimators=50, random_state=42)
    scores = cross_val_score(gb, Xe, ye, cv=min(5, len(Xe)//4), scoring='accuracy')
    accuracies.append(scores.mean())

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(frame_counts, [a * 100 for a in accuracies], 'o-', color='#2196F3', linewidth=2, markersize=8)
ax.axvline(10, color='red', linestyle='--', alpha=0.5, label='10 frames (optimal trade-off)')
ax.set_xlabel('Frames used for classification')
ax.set_ylabel('Accuracy (%)')
ax.set_title('Classification Accuracy vs Number of Frames Used')
ax.set_ylim(0, 105)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../reports/early_detection_accuracy.png', bbox_inches='tight')
plt.show()

best_idx = np.argmax(accuracies)
print(f'\nKey finding: {frame_counts[best_idx]} frames achieves max accuracy of {accuracies[best_idx]:.1%}')
print(f'At 10 frames: {accuracies[frame_counts.index(10)]:.1%} accuracy')
print(f'At 10 fps, 10 frames = 0.33 seconds — fast enough for real-time feedback!')